In [ ]:
import numpy as np
import pandas as pd
import pickle
import re

In [24]:
# Load price prediction pipeline
pipe = pickle.load(open("pipe.pkl", "rb"))

sentiment_model = pickle.load(open("models/sentiment_model.pkl", "rb"))
tfidf = pickle.load(open("models/tfidf.pkl", "rb"))



c:\Users\basav\PYTHON_PRACTICE\venv\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator OneHotEncoder from version 1.5.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\basav\PYTHON_PRACTICE\venv\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator FunctionTransformer from version 1.5.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\basav\PYTHON_PRACTICE\venv\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator ColumnTransformer from version 1.5.1 when using ve

In [25]:
def clean_text(t: str) -> str:
    t = str(t).lower()
    t = re.sub(r'[^a-z0-9\s]', ' ', t)
    t = re.sub(r'\s+', ' ', t).strip()
    return t


In [32]:
def get_sentiment_score(reviews: list[str]) -> float:
    """
    reviews: list of raw review strings for a specific laptop
    returns: average positive sentiment probability in [0,1]
    """
    if not reviews:
        # No reviews -> neutral
        return 0.5
    
    cleaned = [clean_text(r) for r in reviews]
    X = tfidf.transform(cleaned)
    probs = sentiment_model.predict_proba(X)[:, 1]  # positive class
    return float(probs.mean())


In [34]:
sample_reviews = [
    "This laptop is amazing, very fast and great battery life"]

get_sentiment_score(sample_reviews)


0.9350688432718904

In [35]:
def get_fair_price_from_specs(specs: dict) -> float:
    """
    specs: dict with SAME keys as the features used for training, e.g.:

    specs = {
        'Company': 'Dell',
        'TypeName': 'Notebook',
        'Ram': 8,
        'Weight': 1.8,
        'Touchscreen': 0,
        'Ips': 1,
        'ppi': 150.0,
        'Cpu brand': 'Intel Core i5',
        'HDD': 0,
        'SSD': 512,
        'Gpu brand': 'Intel',
        'os': 'Windows 10'
    }

    Returns:
        fair_price (float) on original Rupee scale.
    """
    X = pd.DataFrame([specs])          # single row
    pred_log = pipe.predict(X)[0]      # model trained on log1p(Price)
    fair_price = np.expm1(pred_log)    # back to normal price
    return float(fair_price)


In [36]:
specs_example = {
    'Company': 'Dell',
    'TypeName': 'Notebook',
    'Ram': 8,
    'Weight': 1.8,
    'Touchscreen': 0,
    'Ips': 1,
    'ppi': 150.0,
    'Cpu brand': 'Intel Core i5',
    'HDD': 0,
    'SSD': 512,
    'Gpu brand': 'Intel',
    'os': 'Windows'   # or whatever categories your model saw
}

fair_price = get_fair_price_from_specs(specs_example)
fair_price


67104.48260525725

In [37]:
def evaluate_deal(specs: dict, actual_price: float, reviews: list[str]) -> dict:
    """
    specs: laptop specs dict
    actual_price: current listing price (float)
    reviews: list of review strings for this laptop
    
    returns: dict with all metrics + a textual verdict
    """
    fair_price = get_fair_price_from_specs(specs)
    sentiment_score = get_sentiment_score(reviews)
    
    if actual_price <= 0:
        raise ValueError("actual_price must be > 0")
    
    price_ratio = fair_price / actual_price  # >1 => cheaper than fair; <1 => overpriced
    worthiness = sentiment_score * price_ratio
    
    # Simple rule-based interpretation
    if worthiness >= 0.9:
        verdict = "Excellent deal ✅"
    elif worthiness >= 0.75:
        verdict = "Good deal 🙂"
    elif worthiness >= 0.6:
        verdict = "Average / think twice 😐"
    else:
        verdict = "Not worth it ❌"
    
    return {
        "fair_price": fair_price,
        "actual_price": actual_price,
        "sentiment_score": sentiment_score,
        "price_ratio": price_ratio,
        "worthiness": worthiness,
        "verdict": verdict
    }


In [38]:
reviews_example = [
    "This laptop is amazing, very fast and great battery life"
]

result = evaluate_deal(
    specs=specs_example,
    actual_price=65000.0,
    reviews=reviews_example
)

result


{'fair_price': 67104.48260525725,
 'actual_price': 65000.0,
 'sentiment_score': 0.9350688432718904,
 'price_ratio': 1.032376655465496,
 'worthiness': 0.9653432450470244,
 'verdict': 'Excellent deal ✅'}